# NumPy Statistical and Math Functions

> Generated from [`05-Statistical-And-Math-Functions.py`](05-Statistical-And-Math-Functions.py) by `tools/build_notebooks.py` - edit the `.py` (or its `.md` notes), not this notebook. Changes made here are overwritten on the next build.

## Quick notes

### What this covers
Reducing arrays with `sum`/`mean`/`std`/`var`/`min`/`max`, understanding what `axis=` actually does, running totals with `cumsum`/`cumprod`, sorting with `sort`/`argsort`, locating extremes with `argmax`/`argmin`, and both forms of `np.where`.

### Key points
- Without `axis=`, reductions (`sum`, `mean`, `std`, `var`, `min`, `max`) collapse the whole array to one scalar.
- **`axis=N` means "this axis disappears from the result."** For a `(2, 3)` matrix: `sum(axis=0)` removes axis 0 (rows) and gives one value **per column** (shape `(3,)`); `sum(axis=1)` removes axis 1 (columns) and gives one value **per row** (shape `(2,)`).
- `cumsum`/`cumprod` return a running total/product, same size as the input.
- `np.sort(arr)` returns a new sorted array; `arr.sort()` sorts in place and returns `None`. `np.argsort(arr)` returns the indices that would sort it — useful for sorting one array by another's order.
- `argmax`/`argmin` return the **index** of the extreme value, not the value itself.
- `np.where(condition)` (1-arg) returns a tuple of index arrays (like `nonzero`) — not the matching values. `np.where(condition, if_true, if_false)` (3-arg) is a vectorized ternary, building a new array elementwise.

### Gotchas (the part worth revising twice)
- Mixing up axis 0 vs axis 1 is the single most common NumPy confusion — remember: the axis you name is the one that gets collapsed away, not the one that survives.
- `np.where(condition)` alone gives you indices, not values — you still need `data[np.where(condition)]` (or just `data[condition]`) to get the actual matching values.
- `np.sort` vs `.sort()`: one is non-mutating (safe, returns new array), the other mutates in place and returns `None` — assigning `x = arr.sort()` gives you `None`, not the sorted array.

### When it matters
Per-column vs per-row aggregation (axis) comes up in almost every real dataset — averaging feature columns, summing per-sample rows. `np.where`'s 3-arg form is the standard way to build derived columns/labels without writing a loop.

### One-line recap example
```python
import numpy as np
m = np.array([[1, 2], [3, 4]])
print(m.sum(axis=0))              # [4, 6] - one sum per column
print(np.where(m > 2, m, 0))      # [[0, 0], [3, 4]] - keep, else 0
```

In [ ]:
# 01. sum / mean / std / var / min / max
# 02. The axis Parameter Explained
# 03. cumsum / cumprod
# 04. sort / argsort
# 05. argmax / argmin
# 06. np.where - Both Forms

import numpy as np

## 01. sum / mean / std / var / min / max

- Without axis=, these reduce the ENTIRE array to a single scalar.
- std (standard deviation) and var (variance) measure spread around the mean.

In [ ]:
matrix = np.array([[1, 2, 3], [4, 5, 6]])
print("matrix:\n", matrix)
print("matrix.sum()  =", matrix.sum(), "(all 6 elements added)")
print("matrix.mean() =", matrix.mean())
print("matrix.std()  =", matrix.std(), "(spread around the mean)")
print("matrix.var()  =", matrix.var(), "(std squared)")
print("matrix.min()  =", matrix.min(), ", matrix.max() =", matrix.max())

## 02. The axis Parameter Explained

- axis=0 collapses DOWN the rows -> one result PER COLUMN.
- axis=1 collapses ACROSS the columns -> one result PER ROW.
- Mental model: "axis=N" means "this axis disappears from the output shape".
  matrix.shape is (2, 3). sum(axis=0) removes axis 0 -> result shape (3,).
  sum(axis=1) removes axis 1 -> result shape (2,).

In [ ]:
print("matrix (shape", matrix.shape, "):\n", matrix)
print("# axis=0 -> collapses ROWS, one value per COLUMN (result shape (3,))")
print("matrix.sum(axis=0) =", matrix.sum(axis=0), "(col0: 1+4=5, col1: 2+5=7, col2: 3+6=9)")
print("# axis=1 -> collapses COLUMNS, one value per ROW (result shape (2,))")
print("matrix.sum(axis=1) =", matrix.sum(axis=1), "(row0: 1+2+3=6, row1: 4+5+6=15)")
print("matrix.mean(axis=0) =", matrix.mean(axis=0), "(per-column mean)")
print("matrix.mean(axis=1) =", matrix.mean(axis=1), "(per-row mean)")

## 03. cumsum / cumprod

- Cumulative sum/product: each output element is the running total up to
  and including that position. Output has the SAME size as the input
  (flattened, unless axis= is given).

In [ ]:
seq = np.array([1, 2, 3, 4])
print("seq =", seq)
print("np.cumsum(seq)  =", np.cumsum(seq), "(1, 1+2, 1+2+3, 1+2+3+4)")
print("np.cumprod(seq) =", np.cumprod(seq), "(1, 1*2, 1*2*3, 1*2*3*4)")
print("matrix.cumsum(axis=1):\n", matrix.cumsum(axis=1), "(running total along each row)")

## 04. sort / argsort

- np.sort(arr) returns a NEW sorted array (original untouched);
  arr.sort() sorts IN PLACE and returns None.
- np.argsort(arr) returns the INDICES that would sort the array - useful
  when you need to sort one array based on the order of another.

In [ ]:
unsorted = np.array([3, 1, 4, 1, 5, 9, 2])
print("unsorted =", unsorted)
print("np.sort(unsorted)     =", np.sort(unsorted), "(new array, original untouched)")
print("unsorted after sort() call =", unsorted, "(unchanged)")

sort_indices = np.argsort(unsorted)
print("np.argsort(unsorted)  =", sort_indices, "(indices that WOULD sort it)")
print("unsorted[sort_indices] =", unsorted[sort_indices], "(applying those indices sorts it)")

names = np.array(["c", "a", "d", "b"])
scores = np.array([70, 90, 60, 85])
order = np.argsort(scores)[::-1]  # descending by score
print("\nnames  =", names, ", scores =", scores)
print("Sort names by descending score using argsort(scores)[::-1] ->", names[order])

## 05. argmax / argmin

- Return the INDEX of the max/min value, not the value itself.
- With axis=, return one index per row/column, following the same
  collapse rule as sum/mean.

In [ ]:
print("unsorted =", unsorted)
print("np.argmax(unsorted) =", np.argmax(unsorted), "-> value", unsorted[np.argmax(unsorted)])
print("np.argmin(unsorted) =", np.argmin(unsorted), "-> value", unsorted[np.argmin(unsorted)])
print("matrix:\n", matrix)
print("matrix.argmax(axis=0) =", matrix.argmax(axis=0), "(row-index of max in each column)")
print("matrix.argmax(axis=1) =", matrix.argmax(axis=1), "(col-index of max in each row)")

## 06. np.where - Both Forms

- 1-arg form: np.where(condition) returns a TUPLE of index arrays where
  condition is True (like np.nonzero) - NOT the values themselves.
- 3-arg form: np.where(condition, if_true, if_false) returns an array
  built elementwise: if_true's value where condition holds, if_false's
  value elsewhere - this is the "vectorized ternary" form.
- These two forms are commonly confused because they share a name but
  return completely different things.

In [ ]:
data = np.array([3, 8, 1, 9, 4, 7])
print("data =", data)

indices = np.where(data > 5)
print("np.where(data > 5)             =", indices, "(1-arg form: TUPLE of index arrays, not values)")
print("data[np.where(data > 5)]       =", data[indices], "(use the indices to fetch actual values)")

selected = np.where(data > 5, data, 0)
print("np.where(data > 5, data, 0)    =", selected, "(3-arg form: keep value if True, else 0)")

labels = np.where(data > 5, "high", "low")
print("np.where(data > 5, 'high', 'low') =", labels, "(vectorized ternary, any dtype)")

print("\n# End of Statistical and Math Functions")